## Analysis of the PREACT-digital study sample

In [1]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc
import os
import glob
import numpy as np
import pickle
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import (
    datapath,
    proj_sheet,
    preprocessed_path, redcap_path,
    raw_path,
    backup_path,
)

In [2]:
with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
with open(redcap_path + '/redcap_data.pkl', 'rb') as file:
    df_redcap = pickle.load(file)
with open(preprocessed_path + '/passive_adherence_counts.pkl', 'rb') as file:
    df_pass_adherence = pickle.load(file)
    
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")

split_path = redcap_path + "/sp1_cleaned" +  "/111025_subject_splits.csv"
df_split = pd.read_csv(split_path)



### Check study timeframes

In [ ]:

t5_path_zert = redcap_path + "/t5" + "/Zertifizierung_FOR_Verlaufsdoku_T5.csv"
t5_path = redcap_path + "/t5" + "/FOR_Verlaufsdoku_T5.csv"

t5_zert = pd.read_csv(t5_path_zert, low_memory="False")
t5 = pd.read_csv(t5_path, low_memory="False")

In [ ]:
cols_to_keep = ['for_id','t5_session_date', 't1_session_date','t20_session_date','t5_assessment_date','t20_assessment_date','post_assessment_date', 'ic_date']

In [ ]:
t5_short = t5[cols_to_keep]
t5_zert_short = t5_zert[cols_to_keep]
t5_final = pd.concat([t5_zert_short, t5_short], ignore_index=True)

In [ ]:
t5_final = t5_final.drop_duplicates()

In [ ]:
# (optional) ensure it's a datetime so sorting behaves correctly
t5_final['t5_assessment_date'] = pd.to_datetime(t5_final['t5_assessment_date'], errors='coerce')

dedup = (
    t5_final.sort_values(['for_id', 't5_assessment_date'], na_position='first')  # NaNs first
      .drop_duplicates(subset='for_id', keep='last')                       # keep latest non-null
      .reset_index(drop=True)
)


### Check baseline demographics 

In [ ]:
df_redcap_final["ema_participation"] = (
    df_redcap_final["ema_start_date"].notna().map({True: "yes", False: "no"})
)

In [ ]:
df_redcap_final["ema_participation"].value_counts(dropna=False)

In [ ]:
df_table = df_redcap_final[['age',
 'gender', 'graduation',
 'profession',
 'years_of_education', 'employability', 'scid_cv_prim_cat','ses',
 'bdi_0_sum',
 'asi_0_total',
 'pass_0_sum',
 'mos_0_mean',
 'bsi_0_gsi',
 'bsi_0_sum', 'ema_participation', 'ema_smartphone']]

### Check passive data availability

In [ ]:


# ---------- config ----------
df = df_table.copy()
group_var = "ema_participation"
categorical = ["gender", "graduation", "profession", "employability", "scid_cv_prim_cat"]
continuous = [
    "age", "years_of_education", "bdi_0_sum", "asi_0_total",
    "pass_0_sum", "mos_0_mean", "bsi_0_gsi", "bsi_0_sum", "ses"
]

# ---------- helpers ----------
def fix_mojibake(v):
    # only attempt fix if typical mojibake chars appear
    if isinstance(v, str) and "Ã" in v:
        try:
            return v.encode("latin1").decode("utf-8")
        except Exception:
            return v
    return v

def fmt_pct(n, d):
    pct = 0.0 if d == 0 else 100 * n / d
    return f"{n} ({pct:.1f}%)"

def fmt_p(p):
    if pd.isna(p):
        return ""
    return "<0.001" if p < 1e-3 else f"{p:.3f}"

def cramers_v_bias_corrected(tab):
    # tab: r x k contingency table (numpy array)
    n = tab.sum()
    if n == 0:
        return np.nan
    chi2, _, _, expected = chi2_contingency(tab, correction=False)
    r, k = tab.shape
    phi2 = chi2 / n
    # Bias correction (Bergsma)
    phi2corr = max(0, phi2 - (k-1)*(r-1)/(n-1)) if n > 1 else 0
    rcorr = r - (r-1)**2/(n-1) if n > 1 else r
    kcorr = k - (k-1)**2/(n-1) if n > 1 else k
    denom = max(1e-12, min(kcorr-1, rcorr-1))
    return np.sqrt(phi2corr / denom)

def smd_cohens_d(a, b):
    a = np.asarray(a); b = np.asarray(b)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    ma, mb = a.mean(), b.mean()
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    # pooled SD
    sp2 = ((len(a)-1)*sa**2 + (len(b)-1)*sb**2) / (len(a)+len(b)-2)
    sp = np.sqrt(sp2) if sp2 > 0 else np.nan
    return (ma - mb)/sp if np.isfinite(sp) and sp > 0 else np.nan

# ---------- normalize inputs ----------
# map group to Yes/No and drop others to ensure exactly 2 groups
group_map = {"yes":"Yes","1":"Yes","true":"Yes","no":"No","0":"No","false":"No"}
df[group_var] = (df[group_var].astype(str).str.strip().str.lower().map(group_map))
df = df[df[group_var].isin(["Yes","No"])].copy()

# fix mojibake in object columns
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].map(fix_mojibake)

# assure types
for c in categorical:
    if c in df.columns:
        df[c] = df[c].astype("category")
for c in continuous:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# group sizes
n_yes = (df[group_var] == "Yes").sum()
n_no  = (df[group_var] == "No").sum()

rows = []
# header row with Ns
rows.append(("n", "", "", f"{n_yes}", f"{n_no}", "", ""))

# ---------- categorical blocks ----------
for var in [c for c in categorical if c in df.columns]:
    miss = df[var].isna().sum()
    sub = df[[group_var, var]].copy()
    # build table with both columns ensured
    tab = pd.crosstab(sub[var], sub[group_var], dropna=True)
    for col in ["Yes","No"]:
        if col not in tab.columns:
            tab[col] = 0
    tab = tab[["Yes","No"]]
    # p-value + effect
    try:
        if tab.shape == (2,2) and ((tab.values < 5).any()):
            # Fisher for sparse 2x2
            _, p = fisher_exact(tab.values)
        else:
            _, p, _, _ = chi2_contingency(tab.values, correction=False)
        v = cramers_v_bias_corrected(tab.values)
    except Exception:
        p, v = np.nan, np.nan
    # first line: variable name with missing & p
    rows.append((f"{var}, n (%)", "—", str(miss), "", "", f"{v:.3f}" if pd.notna(v) else "", fmt_p(p)))
    # each level
    for lvl, r in tab.iterrows():
        y, n = int(r["Yes"]), int(r["No"])
        rows.append(("", str(lvl), "", fmt_pct(y, n_yes), fmt_pct(n, n_no), "", ""))

# ---------- continuous blocks ----------
for var in [c for c in continuous if c in df.columns]:
    miss = df[var].isna().sum()
    y = df.loc[df[group_var]=="Yes", var].dropna()
    n = df.loc[df[group_var]=="No",  var].dropna()
    # medians & IQRs
    def med_iqr(x):
        if len(x)==0:
            return "—"
        q1, q2, q3 = np.percentile(x, [25,50,75])
        return f"{q2:.1f} [{q1:.1f},{q3:.1f}]"
    yes_txt = med_iqr(y)
    no_txt  = med_iqr(n)
    # p-value Mann–Whitney (two-sided)
    try:
        stat, p = mannwhitneyu(y, n, alternative="two-sided")
    except ValueError:
        p = np.nan
    # effect: Cohen's d (mean-based; robust alt would be Cliff's delta)
    d = smd_cohens_d(y, n)
    rows.append((f"{var}, median [Q1,Q3]", "", str(miss), yes_txt, no_txt,
                 f"{d:.3f}" if pd.notna(d) else "", fmt_p(p)))

# assemble table
table = pd.DataFrame(rows, columns=["Variable","Level","Missing","Yes","No","Effect","P-Value"])

# nicer display order: put header rows (Level='—' or '') right above their levels automatically
# (already constructed in order)

# print & save
print(table.to_markdown(index=False))
table.to_csv("outcomes_jitai/sample_overview_clean.csv", index=False)


### Bias Android/ I-Phone

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

# ---------- config ----------
# (keep this if you only want participants; otherwise remove this line)
df = df_table.copy()
df = df[df["ema_participation"].astype(str).str.lower().isin(["yes","1","true"])]

group_var = "ema_smartphone"  # original column with raw phone info
categorical = ["gender", "graduation", "profession", "employability", "scid_cv_prim_cat"]
continuous = [
    "age", "years_of_education", "bdi_0_sum", "asi_0_total",
    "pass_0_sum", "mos_0_mean", "bsi_0_gsi", "bsi_0_sum", "ses"
]

# ---------- helpers ----------
def fix_mojibake(v):
    if isinstance(v, str) and "Ã" in v:
        try:
            return v.encode("latin1").decode("utf-8")
        except Exception:
            return v
    return v

def fmt_pct(n, d):
    pct = 0.0 if d == 0 else 100 * n / d
    return f"{n} ({pct:.1f}%)"

def fmt_p(p):
    if pd.isna(p):
        return ""
    return "<0.001" if p < 1e-3 else f"{p:.3f}"

def cramers_v_bias_corrected(tab):
    n = tab.sum()
    if n == 0:
        return np.nan
    chi2, _, _, expected = chi2_contingency(tab, correction=False)
    r, k = tab.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - (k-1)*(r-1)/(n-1)) if n > 1 else 0
    rcorr = r - (r-1)**2/(n-1) if n > 1 else r
    kcorr = k - (k-1)**2/(n-1) if n > 1 else k
    denom = max(1e-12, min(kcorr-1, rcorr-1))
    return np.sqrt(phi2corr / denom)

def smd_cohens_d(a, b):
    a = np.asarray(a); b = np.asarray(b)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    ma, mb = a.mean(), b.mean()
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    sp2 = ((len(a)-1)*sa**2 + (len(b)-1)*sb**2) / (len(a)+len(b)-2)
    sp = np.sqrt(sp2) if sp2 > 0 else np.nan
    return (ma - mb)/sp if np.isfinite(sp) and sp > 0 else np.nan

# ---------- normalize the phone groups to exactly: Android / iPhone ----------
# make a cleaned string version
s = df[group_var].astype(str).str.strip().str.lower()
# remove spaces/hyphens so 'i phone'/'i-phone' becomes 'iphone'
s = s.str.replace(r'[\s-]+', '', regex=True)

def to_phone_group(v: str):
    if any(k in v for k in ["android", "pixel", "samsung", "huawei", "xiaomi", "oneplus", "oppo", "vivo"]):
        return "Android"
    if any(k in v for k in ["ios", "iphone", "apple"]):
        return "iPhone"
    return np.nan  # unknown/other

df["phone_group"] = s.map(to_phone_group)
df = df[df["phone_group"].isin(["Android", "iPhone"])].copy()

# fix mojibake in object columns (for display)
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].map(fix_mojibake)

# assure types
for c in categorical:
    if c in df.columns:
        df[c] = df[c].astype("category")
for c in continuous:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# group sizes
n_android = (df["phone_group"] == "Android").sum()
n_iphone  = (df["phone_group"] == "iPhone").sum()

rows = []
# header row with Ns
rows.append(("n", "", "", f"{n_android}", f"{n_iphone}", "", ""))

col_order = ["Android", "iPhone"]

# ---------- categorical blocks ----------
for var in [c for c in categorical if c in df.columns]:
    miss = df[var].isna().sum()
    sub = df[["phone_group", var]].copy()
    tab = pd.crosstab(sub[var], sub["phone_group"], dropna=True)
    for col in col_order:
        if col not in tab.columns:
            tab[col] = 0
    tab = tab[col_order]
    # p-value + effect
    try:
        if tab.shape == (2,2) and ((tab.values < 5).any()):
            _, p = fisher_exact(tab.values)
        else:
            _, p, _, _ = chi2_contingency(tab.values, correction=False)
        v = cramers_v_bias_corrected(tab.values)
    except Exception:
        p, v = np.nan, np.nan
    rows.append((f"{var}, n (%)", "—", str(miss), "", "", f"{v:.3f}" if pd.notna(v) else "", fmt_p(p)))
    for lvl, r in tab.iterrows():
        a, i = int(r["Android"]), int(r["iPhone"])
        rows.append(("", str(lvl), "", fmt_pct(a, n_android), fmt_pct(i, n_iphone), "", ""))

# ---------- continuous blocks ----------
for var in [c for c in continuous if c in df.columns]:
    miss = df[var].isna().sum()
    a = df.loc[df["phone_group"]=="Android", var].dropna()
    i = df.loc[df["phone_group"]=="iPhone",  var].dropna()
    def med_iqr(x):
        if len(x)==0:
            return "—"
        q1, q2, q3 = np.percentile(x, [25,50,75])
        return f"{q2:.1f} [{q1:.1f},{q3:.1f}]"
    a_txt = med_iqr(a)
    i_txt = med_iqr(i)
    try:
        _, p = mannwhitneyu(a, i, alternative="two-sided")
    except ValueError:
        p = np.nan
    d = smd_cohens_d(a, i)
    rows.append((f"{var}, median [Q1,Q3]", "", str(miss), a_txt, i_txt,
                 f"{d:.3f}" if pd.notna(d) else "", fmt_p(p)))

# assemble & save
table = pd.DataFrame(rows, columns=["Variable","Level","Missing","Android","iPhone","Effect","P-Value"])
print(table.to_markdown(index=False))
table.to_csv("outcomes_jitai/sample_overview_android_vs_iphone.csv", index=False)


### Passive Data Adherence

In [ ]:
df_monitoring[df_monitoring.dropout_2.notna()]

### Dropout

In [ ]:
# Robust (vermeidet Attributzugriff & SettingWithCopy)
df_monitoring.loc[:, 'dropout_type'] = (
    df_monitoring['dropout_1'].combine_first(df_monitoring['dropout_2'])
)
df_monitoring.drop(["dropout_1", "dropout_2"], axis=1, inplace=True)

In [ ]:
df_dropout = df_monitoring[df_monitoring.dropout_type.notna()]

In [ ]:
df_redcap_ema = df_redcap_final.loc[df_redcap_final.ema_participation == "yes"]

In [ ]:
df_redcap_ema = df_redcap_ema.merge(df_monitoring, on="for_id", how="right")

In [ ]:
df_dropout.groupby("status_short")["customer"].count()

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# --- 1) Aggregate from your dataframe ---
series = (
    df_redcap_ema
    .groupby("status_short")["customer"]
    .nunique()
    .sort_values(ascending=False)         # keep if you want largest at bottom
)

labels = series.index.tolist()
counts = series.values.tolist()
total  = int(series.sum())
percs  = [c / total * 100 for c in counts]

# --- 2) Color assignment: YlGnBu with explicit pins ---
colorscale = "YlGnBu"

# helper to find indices case-insensitively
def find_idx(substr):
    s = substr.casefold()
    for i, lab in enumerate(labels):
        if s in lab.casefold():
            return i
    return None

idx_dropout       = find_idx("dropout")
idx_abgeschlossen = find_idx("abgeschlossen")

# sample helper
def cs(stop):
    return px.colors.sample_colorscale(colorscale, [stop])[0]

# choose stops (tuned to get light green for dropout, dark blue for abgeschlossen)
light_green = cs(0.38)   # ~light green region of YlGnBu
dark_blue   = cs(0.95)   # dark blue end
colors = [None] * len(labels)

if idx_dropout is not None:
    colors[idx_dropout] = light_green
if idx_abgeschlossen is not None:
    colors[idx_abgeschlossen] = dark_blue

# fill remaining categories with mid-range tones spaced evenly
remaining = [i for i in range(len(labels)) if colors[i] is None]
for i, stop in zip(remaining, np.linspace(0.50, 0.85, len(remaining))):
    colors[i] = cs(stop)

# --- 3) Build stacked bar (single column) ---
fig = go.Figure()
for label, count, perc, col in zip(labels, counts, percs, colors):
    fig.add_bar(
        x=["Status"], y=[count], name=label,
        marker_color=col,
        text=[f"{count} ({perc:.1f}%)"],
        textposition="inside",
        hovertemplate=f"{label}<br>N = {count} ({perc:.1f}%)<extra></extra>",
    )

# dotted separators
shapes, cum = [], 0
for c in counts[:-1]:
    cum += c
    shapes.append(dict(
        type="line", xref="paper", yref="y",
        x0=0.05, x1=0.95, y0=cum, y1=cum,
        line=dict(width=1, dash="dot", color="rgba(0,0,0,0.35)")
    ))

# --- 4) Layout ---
fig.update_layout(
    barmode="stack",
    template="simple_white",
    width=560, height=820,
    margin=dict(l=60, r=200, t=80, b=40),
    font=dict(family="Arial, DejaVu Sans, Sans-Serif", size=20),
    title=dict(text=f"Teilnehmenden Status • Total N = {total}", x=0.0, font=dict(size=22)),
    legend=dict(x=1.02, y=0.5, yanchor="middle", bordercolor="rgba(0,0,0,0.1)", borderwidth=1),
    shapes=shapes,
    yaxis=dict(title="", gridcolor="rgba(0,0,0,0.08)", zeroline=False),
    xaxis=dict(title="", showticklabels=False),
)
fig.update_traces(texttemplate="%{text}", textfont_size=20, cliponaxis=False)

fig.show()
# fig.write_html("status_plot.html", include_plotlyjs="cdn")
# fig.write_image("status_plot.png", scale=2)  # needs `pip install -U kaleido`


### Adherence

In [ ]:
df_adherence = pd.merge(df_pass_adherence, df_redcap_ema, left_on="id", right_on="customer")

In [ ]:
df_adherence_filtered = df_adherence[df_adherence.study_version.isin(["Lang","Lang (Wechsel)"])]

In [ ]:
df_adherence_filtered = df_adherence_filtered.loc[df_adherence_filtered.modality.isin(['Latitude', 'Steps','HeartRate','ActivityType'])]


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# -------------------- CONFIG --------------------
ID_COL      = "id"             # change if needed
DATE_COL    = "start_date"
BASE_COL    = "ema_base_start"
MOD_COL     = "modality"
POINTS_COL  = "n_data_points"

MAX_DAYS    = 100              # first 100 days: 0..99
DENOM       = "all"            # "all" or "per_modality"
ROLL        = 7                # 7-day rolling mean; set to 0/None to disable
TOP_K       = None             # e.g., 10 to keep only top-K modalities by mean coverage

# Your dataframe here:
df = df_adherence_filtered.copy()             # <-- replace df_all with your DataFrame variable

# -------------------- 1) Parse & filter --------------------
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", utc=True)
df[BASE_COL] = pd.to_datetime(df[BASE_COL], errors="coerce", utc=True)

# Keep only participants with a valid baseline
df = df[df[BASE_COL].notna()].copy()

# Dtypes + numeric
df[ID_COL]     = df[ID_COL].astype("category")
df[MOD_COL]    = df[MOD_COL].astype("category")
df[POINTS_COL] = pd.to_numeric(df[POINTS_COL], errors="coerce").fillna(0)

# Relative study day
df["relative_day"] = (df[DATE_COL] - df[BASE_COL]).dt.floor("D").dt.days
df = df[(df["relative_day"] >= 0) & (df["relative_day"] < MAX_DAYS)].copy()

# -------------------- 2) Any data (>0) per id × modality × day --------------------
pres = (
    df.assign(has_data=(df[POINTS_COL] > 0).astype(np.int8))
      .groupby([MOD_COL, ID_COL, "relative_day"], observed=True)["has_data"]
      .max()
      .reset_index()
)

# Numerator: #ids with data per modality × day
num = (
    pres.groupby([MOD_COL, "relative_day"], observed=True)["has_data"]
        .sum()
        .reset_index(name="n_ids")
)

# Denominator
if DENOM == "per_modality":
    denom = (pres.groupby(MOD_COL, observed=True)[ID_COL]
                 .nunique()
                 .rename("denom")
                 .reset_index())
    availability = num.merge(denom, on=MOD_COL, how="left")
else:  # "all"
    total_ids = df[ID_COL].nunique()
    availability = num.assign(denom=total_ids)

# Fill missing days with 0 so lines don’t break
mods = availability[MOD_COL].unique()
all_days = np.arange(0, MAX_DAYS, dtype=int)
availability = (
    availability.set_index([MOD_COL, "relative_day"])
                .reindex(pd.MultiIndex.from_product([mods, all_days],
                                                    names=[MOD_COL, "relative_day"]),
                         fill_value=0)
                .reset_index()
)

# Percentage
availability["percentage"] = 100 * availability["n_ids"] / availability["denom"]

# Keep only top-K modalities (optional)
if TOP_K:
    keep = (availability.groupby(MOD_COL)["percentage"]
                        .mean()
                        .sort_values(ascending=False)
                        .head(TOP_K).index)
    availability = availability[availability[MOD_COL].isin(keep)].copy()

# Smooth (optional)
if ROLL and ROLL > 1:
    availability = availability.sort_values([MOD_COL, "relative_day"])
    availability["percentage"] = (
        availability.groupby(MOD_COL)["percentage"]
                    .transform(lambda s: s.rolling(ROLL, center=True, min_periods=1).mean())
    )

# Rename for plotting
availability = availability.rename(columns={
    MOD_COL: "type",
    "relative_day": "relative_day"
})

# -------------------- 3) Plotly line plot --------------------
fig = px.line(
    availability,
    x="relative_day",
    y="percentage",
    color="type",
    labels={
        "relative_day": "Study Day",
        "percentage": "Data Availability (%)",
        "type": "Modality"
    },
    title=""
)

# Layout like your example
fig.update_layout(
    width=1000,
    height=500,
    font=dict(size=16),
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
    legend_title_font=dict(size=16),
    legend_font=dict(size=14),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

# Dotted vertical line at day 14 + annotation
fig.add_shape(
    dict(
        type="line",
        x0=14, x1=14, y0=0, y1=1,
        xref="x", yref="paper",
        line=dict(color="black", width=2, dash="dot")
    )
)
# --- put the dotted line under the curves ---
fig.add_shape(
    dict(
        type="line",
        x0=14, x1=14, y0=0, y1=1,
        xref="x", yref="paper",
        line=dict(color="black", width=2, dash="dot"),
        layer="below"
    )
)

# --- annotation ABOVE the plot (no overlap) ---
fig.add_annotation(
    x=14, xref="x",
    y=1.06, yref="paper",             # above the plotting area
    text="End of first active assessment phase",
    showarrow=True, arrowhead=1,
    ax=0, ay=-30,                      # arrow points down into the plot
    xanchor="center", yanchor="bottom",
    font=dict(size=16)
)

# --- axes & margins ---
# Extend x-axis a hair to the left so the "0" tick isn't glued to the left border,
# and show ticks up to 100.
fig.update_xaxes(
    range=[-0.5, MAX_DAYS],                          # shows 0…100 ticks; lines stop at 99 naturally
    tickmode="array",
    tickvals=list(range(0, MAX_DAYS + 1, 10)),       # 0,10,...,100
    ticks="outside",
    ticklen=6,
    title_standoff=16
)

# Start y at 40, keep nice 5% ticks
# Extra left margin so y tick labels aren't crowded; outside-left placement for clarity.
ymax = float(np.ceil(availability["percentage"].max() / 5) * 5)
fig.update_yaxes(
    range=[35, ymax],
    tickmode="linear",
    dtick=10,
    ticks="outside",
    ticklen=6,
    ticklabelposition="outside left",
    title_standoff=16
)

# More top margin so the above-plot annotation never gets clipped
fig.update_layout(
    margin=dict(l=90, r=20, t=110, b=80)
)


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# -------------------- CONFIG --------------------
ID_COL, DATE_COL, BASE_COL = "id", "start_date", "ema_base_start"
MOD_COL, TYPE_COL = "modality", "type"      # we'll try both for 'Latitude'
PHONE_COL = "ema_smartphone"                # values: "Android" or "I-Phone"

TARGET_LABEL = "Latitude"
MAX_DAYS = 100               # show days 0..99; ticks 0..100
ROLL = 7                     # 7-day rolling mean; set 0/None to disable
DENOM_MODE = "ever_latitude" # "all_group" or "ever_latitude"

df = df_adherence_filtered.copy()

# --- parse dates & baseline filter ---
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", utc=True)
df[BASE_COL] = pd.to_datetime(df[BASE_COL], errors="coerce", utc=True)
df = df[df[BASE_COL].notna()].copy()

# --- keep only exact phone groups (no normalization) ---
df = df[df[PHONE_COL].isin(["Android", "I-Phone"])].copy()

# --- find the column that contains 'Latitude' label ---
label_col = None
if MOD_COL in df.columns and TARGET_LABEL in df[MOD_COL].astype(str).unique():
    label_col = MOD_COL
elif TYPE_COL in df.columns and TARGET_LABEL in df[TYPE_COL].astype(str).unique():
    label_col = TYPE_COL
else:
    # if neither contains "Latitude", bail early with an empty plot input
    df = df.iloc[0:0].copy()

# --- filter to Latitude rows only ---
if label_col is not None:
    df = df[df[label_col].astype(str) == TARGET_LABEL].copy()

# --- relative day 0..99 ---
df["relative_day"] = (df[DATE_COL] - df[BASE_COL]).dt.floor("D").dt.days
df = df[(df["relative_day"] >= 0) & (df["relative_day"] < MAX_DAYS)].copy()

# --- presence per user-day (robust to missing n_data_points) ---
day_counts = (
    df.groupby([PHONE_COL, ID_COL, "relative_day"], observed=True)
      .size()
      .reset_index(name="n_rows")
)
num = (
    day_counts.assign(has_data=(day_counts["n_rows"] > 0).astype(np.int8))
              .groupby([PHONE_COL, "relative_day"], observed=True)["has_data"]
              .sum()
              .reset_index(name="n_ids")
)

# --- denominator per phone group ---
if DENOM_MODE == "ever_latitude":
    # only users in that phone group who ever had at least one Latitude row within 0..99
    denom = (
        day_counts.groupby(PHONE_COL, observed=True)[ID_COL]
                  .nunique()
                  .rename("denom")
                  .reset_index()
    )
else:  # "all_group"
    denom = (
        df.drop_duplicates([ID_COL, PHONE_COL])
          .groupby(PHONE_COL, observed=True)[ID_COL]
          .nunique()
          .rename("denom")
          .reset_index()
    )

availability = num.merge(denom, on=PHONE_COL, how="left")

# --- full day grid & percentage ---
phones = availability[PHONE_COL].unique()
all_days = np.arange(0, MAX_DAYS, dtype=int)
availability = (
    availability.set_index([PHONE_COL, "relative_day"])
                .reindex(pd.MultiIndex.from_product([phones, all_days],
                                                    names=[PHONE_COL, "relative_day"]),
                         fill_value=0)
                .reset_index()
)
availability["percentage"] = 100.0 * availability["n_ids"] / availability["denom"].replace({0: np.nan})
availability["percentage"] = availability["percentage"].fillna(0)

# --- optional smoothing ---
if ROLL and ROLL > 1:
    availability = availability.sort_values([PHONE_COL, "relative_day"])
    availability["percentage"] = (
        availability.groupby(PHONE_COL)["percentage"]
                    .transform(lambda s: s.rolling(ROLL, center=True, min_periods=1).mean())
    )

# --- plot ---
fig = px.line(
    availability,
    x="relative_day",
    y="percentage",
    color=PHONE_COL,
    labels={
        "relative_day": "Study Day",
        "percentage": "GPS Availability (%)",
        PHONE_COL: "Smartphone"
    },
    title=""
)

# day 14 marker (below lines) + annotation above plot
fig.add_shape(dict(type="line", x0=14, x1=14, y0=0, y1=1, xref="x", yref="paper",
                   line=dict(color="black", width=2, dash="dot"), layer="below"))
fig.add_annotation(
    x=14, xref="x", y=1.06, yref="paper",
    text="End of first active assessment phase",
    showarrow=True, arrowhead=1, ax=0, ay=-30,
    xanchor="center", yanchor="bottom", font=dict(size=16)
)

# axes
fig.update_xaxes(
    range=[-0.5, MAX_DAYS],  # ticks 0..100, lines stop at 99
    tickmode="array", tickvals=list(range(0, MAX_DAYS + 1, 10)),
    ticks="outside", ticklen=6, title_standoff=16
)

# smart y-limits: don’t hide tiny lines
ymax_raw = availability["percentage"].max()
ymax = float(np.ceil((ymax_raw if np.isfinite(ymax_raw) else 10) / 10) * 10)
ymin = 0  # since lines are near x-axis
if ymax <= 5:
    ymax = 5  # ensure visible span

fig.update_yaxes(
    range=[15, ymax],
    tickmode="linear",
    dtick=10,
    ticks="outside",
    ticklen=6,
    ticklabelposition="outside left",
    title_standoff=16
)

fig.update_layout(
    width=1000, height=500,
    font=dict(size=16),
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
    legend_title_font=dict(size=16),
    legend_font=dict(size=14),
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=90, r=20, t=110, b=80)
)

fig.show()

# Optional: quick sanity prints
print("Denominator mode:", DENOM_MODE)
print("Denominators per group:", availability.groupby(PHONE_COL)["denom"].max().to_dict())
print(availability["percentage"].describe())
